In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [11]:
def run_drone_simulation(kp_pos, ki_pos, kd_pos, wind_gust_amp, noise_std):
    """
    Simulates a 2D quadrotor UAV tracking a Figure-8 (Lemniscate) parametric trajectory.

    Drone Dynamics:
    - Mass m = 0.5 kg
    - Parametric Figure-8: X(t) = A * sin(w * t), Y(t) = A * sin(w * t) * cos(w * t)
    """

    # --- Drone Physical Constants ---
    m = 0.5          # Quadcopter mass (kg)
    b = 0.25         # Drag/Damping coefficient (N*s/m)
    t_span = (0, 20) # Total simulation time (seconds)
    dt = 0.01        # Time step (seconds)
    t_eval = np.arange(t_span[0], t_span[1], dt)
    num_steps = len(t_eval)

    # --- Figure-8 Parametric Trajectory Parameters ---
    A = 10.0         # Amplitude (meters)
    omega = 0.4      # Trajectory speed frequency (rad/s)

    # Desired trajectory coordinates: Lemniscate of Gerono
    x_ref = A * np.sin(omega * t_eval)
    y_ref = A * np.sin(omega * t_eval) * np.cos(omega * t_eval)

    # --- Disturbance Profile (Wind & Turbulence) ---
    np.random.seed(42)
    wind_x_noise = np.random.normal(0, noise_std * 0.5, num_steps)
    wind_y_noise = np.random.normal(0, noise_std * 0.5, num_steps)

    # Intermittent Wind Gust Impulses
    gust_x = np.zeros(num_steps)
    gust_y = np.zeros(num_steps)
    gust_x[int(5/dt):int(6.5/dt)] = wind_gust_amp * 2.5   # Lateral gust at t = 5s
    gust_y[int(12/dt):int(13.5/dt)] = -wind_gust_amp * 3.0 # Crosswind gust at t = 12s

    total_wind_x = wind_x_noise + gust_x
    total_wind_y = wind_y_noise + gust_y

    # --- Drone State Initialization ---
    x, y = 0.0, 0.0      # Position
    vx, vy = 0.0, 0.0    # Velocity

    # Integral errors
    ix, iy = 0.0, 0.0
    prev_ex, prev_ey = 0.0, 0.0

    # History Arrays
    x_hist = np.zeros(num_steps)
    y_hist = np.zeros(num_steps)
    e_x_hist = np.zeros(num_steps)
    e_y_hist = np.zeros(num_steps)

    # --- Simulation Loop ---
    for i in range(num_steps):
        # Target positions at current step
        xr, yr = x_ref[i], y_ref[i]

        # Position Errors
        ex = xr - x
        ey = yr - y

        # Integrals with Anti-Windup Clamping
        ix = np.clip(ix + ex * dt, -10.0, 10.0)
        iy = np.clip(iy + ey * dt, -10.0, 10.0)

        # Derivatives
        dex = (ex - prev_ex) / dt
        dey = (ey - prev_ey) / dt
        prev_ex, prev_ey = ex, ey

        # PID Controller Force Demands
        ux = (kp_pos * ex) + (ki_pos * ix) + (kd_pos * dex)
        uy = (kp_pos * ey) + (ki_pos * iy) + (kd_pos * dey)

        # Apply Motor Saturation Limits (N)
        ux = np.clip(ux, -30.0, 30.0)
        uy = np.clip(uy, -30.0, 30.0)

        # Equations of Motion: m * a + b * v = Force_control + Force_wind
        ax = (ux + total_wind_x[i] - b * vx) / m
        ay = (uy + total_wind_y[i] - b * vy) / m

        # Euler Integration
        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

        # Logging
        x_hist[i] = x
        y_hist[i] = y
        e_x_hist[i] = ex
        e_y_hist[i] = ey

    # --- Plotting Subsystem (Visualization) ---
    fig = plt.figure(figsize=(14, 6))

    # Subplot 1: 2D Spatial Figure-8 Trajectory Tracking
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(x_ref, y_ref, 'r--', linewidth=2, label='Target Trajectory (Figure-8)')
    ax1.plot(x_hist, y_hist, 'b-', linewidth=2, label='Actual Drone Path')
    ax1.scatter([x_hist[0]], [y_hist[0]], color='green', s=80, zorder=5, label='Start Point')
    ax1.scatter([x_hist[-1]], [y_hist[-1]], color='purple', marker='X', s=100, zorder=5, label='End Point')
    ax1.set_title('Drone Figure-8 Trajectory Tracking (0.5 kg Quadcopter)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('X Position [m]', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Y Position [m]', fontsize=11, fontweight='bold')
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, linestyle=':', alpha=0.7)
    ax1.axis('equal')

    # Subplot 2: Tracking Errors over Time
    ax2 = fig.add_subplot(1, 2, 2)
    error_magnitude = np.sqrt(e_x_hist**2 + e_y_hist**2)
    ax2.plot(t_eval, e_x_hist, 'g-', label='X Error (m)')
    ax2.plot(t_eval, e_y_hist, 'm-', label='Y Error (m)')
    ax2.plot(t_eval, error_magnitude, 'r:', linewidth=2, label='Total Euclidean Error (m)')
    ax2.set_title('Tracking Errors & Disturbance Response', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Time [s]', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Error Distance [m]', fontsize=11, fontweight='bold')
    ax2.legend(loc='upper right', frameon=True)
    ax2.grid(True, linestyle=':', alpha=0.7)

    plt.tight_layout()
    plt.show()

In [12]:
# --- Interactive UI Dashboard Controls ---
slider_kp = widgets.FloatSlider(value=4.0, min=0.0, max=12.0, step=0.2, description='Kp (Proportional):', style={'description_width': '130px'}, layout=widgets.Layout(width='400px'))
slider_ki = widgets.FloatSlider(value=0.2, min=0.0, max=2.0, step=0.05, description='Ki (Integral):', style={'description_width': '130px'}, layout=widgets.Layout(width='400px'))
slider_kd = widgets.FloatSlider(value=2.5, min=0.0, max=8.0, step=0.2, description='Kd (Derivative):', style={'description_width': '130px'}, layout=widgets.Layout(width='400px'))
slider_gust = widgets.FloatSlider(value=2.0, min=0.0, max=8.0, step=0.5, description='Wind Gust Force:', style={'description_width': '130px'}, layout=widgets.Layout(width='400px'))
slider_noise = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='Turbulence Level:', style={'description_width': '130px'}, layout=widgets.Layout(width='400px'))

In [13]:
btn_run = widgets.Button(description='Simulate Drone', button_style='success', icon='play', layout=widgets.Layout(width='200px', height='40px'))
output_plot = widgets.Output()

In [14]:
def on_button_click(b):
    with output_plot:
        clear_output(wait=True)
        run_drone_simulation(
            kp_pos=slider_kp.value,
            ki_pos=slider_ki.value,
            kd_pos=slider_kd.value,
            wind_gust_amp=slider_gust.value,
            noise_std=slider_noise.value
        )

btn_run.on_click(on_button_click)

In [15]:
# UI Layout Setup
controls_vbox = widgets.VBox([
    widgets.HBox([slider_kp, slider_gust]),
    widgets.HBox([slider_ki, slider_noise]),
    widgets.HBox([slider_kd]),
    widgets.Box([btn_run], layout=widgets.Layout(display='flex', justify_content='center', padding='10px'))
])

In [16]:
display(controls_vbox, output_plot)

Output()

In [10]:
# Run initial simulation
on_button_click(None)